# Exploração dos Dados

# 1. Descrição do Dataset

O conjunto de dados utilizado neste estudo foi construído a partir de imagens multiespectrais provenientes de três diferentes missões de observação da Terra amplamente utilizadas em aplicações de sensoriamento remoto: **Landsat, MODIS e Sentinel**. Essas plataformas orbitais fornecem dados espectrais capazes de capturar características físicas e químicas da superfície terrestre, sendo amplamente utilizadas em estudos geológicos, ambientais e de prospecção mineral.

As imagens foram previamente processadas e convertidas para o formato **GeoTIFF (.tif)**, contendo múltiplas bandas espectrais correspondentes a diferentes intervalos do espectro eletromagnético. Essas bandas permitem identificar propriedades como composição mineral, variações de temperatura, refletância e umidade da superfície.

Para possibilitar a utilização dessas imagens em modelos de aprendizado de máquina, foi realizado um processo de **padronização espacial**, no qual todas as imagens foram ajustadas para dimensões de **256 × 256 pixels**. Quando necessário, as imagens foram recortadas centralmente ou preenchidas com valores nulos (padding) para garantir uniformidade dimensional.

Após esse pré-processamento, cada imagem foi convertida para um formato tabular no qual:

* **Cada imagem corresponde a uma linha do dataset**
* **Cada pixel de cada banda corresponde a uma coluna**
* O dataset contém ainda metadados como:

  * identificador da imagem
  * caminho do arquivo original

Dessa forma, cada linha representa um vetor de características espectrais e espaciais extraídas da imagem.

Formalmente, o dataset pode ser representado como:

X∈RN×(B×P)

onde:

* (N) = número de imagens
* (B) = número de bandas espectrais
* (P = 256 \times 256) = número de pixels por banda

O resultado final consiste em **três datasets independentes**, um para cada sensor:

* **Landsat Dataset**
* **MODIS Dataset**
* **Sentinel Dataset**

Cada dataset preserva as características espectrais específicas do sensor, permitindo análises comparativas entre diferentes resoluções espaciais e espectrais.

Esses dados serão utilizados para treinar modelos de aprendizado de máquina capazes de identificar padrões associados à **presença de minerais de terras raras**, explorando as assinaturas espectrais presentes nas imagens multiespectrais.

---

# 2. Roteiro de Exploração de Dados (EDA)

A etapa de exploração dos dados tem como objetivo compreender as características estatísticas e espectrais do dataset, identificar padrões relevantes e formular hipóteses que possam orientar o treinamento dos modelos de aprendizado de máquina.

A exploração será conduzida em etapas.

---

# Etapa 1 — Verificação estrutural do dataset

Primeiramente, é necessário validar a estrutura dos dados gerados.

Perguntas principais:

* Quantas imagens existem em cada dataset?
* Quantas bandas espectrais existem por sensor?
* Quantas features foram geradas?

Análises:

* número de linhas (imagens)
* número de colunas (features)
* tamanho dos datasets
* proporção entre imagens positivas e negativas (se houver rótulos)

Objetivo:

Garantir que o dataset foi gerado corretamente e compreender sua dimensionalidade.

---

# Etapa 2 — Estatísticas descritivas

Nesta etapa são calculadas estatísticas básicas dos valores dos pixels.

Análises:

* média
* mediana
* desvio padrão
* valores mínimos e máximos
* distribuição dos valores

Isso ajuda a identificar:

* diferenças de escala entre sensores
* presença de valores extremos
* possíveis valores inválidos

Também permite avaliar se será necessário aplicar:

* normalização
* padronização
* clipping

---

# Etapa 3 — Distribuição espectral das bandas

Cada banda espectral captura diferentes propriedades da superfície terrestre.

Análises:

* histogramas por banda
* média por banda
* comparação entre sensores
* variação espectral média

Objetivo:

Identificar quais bandas carregam maior variação informacional.

Isso pode indicar quais bandas são mais relevantes para identificar minerais.

---

# Etapa 4 — Correlação entre bandas

A análise de correlação permite verificar redundância espectral.

Análises:

* matriz de correlação entre bandas
* identificação de bandas altamente correlacionadas
* redução de dimensionalidade potencial

Possíveis técnicas futuras:

* PCA
* seleção de features

Objetivo:

Reduzir redundância e melhorar eficiência do modelo.

---

# Etapa 5 — Visualização espacial das imagens

Mesmo com o dataset transformado em formato tabular, é importante reconstruir imagens para inspeção visual.

Análises:

* reconstrução de imagens a partir do vetor
* visualização de bandas individuais
* composição RGB

Objetivo:

Verificar se os dados mantêm coerência espacial após o processamento.

---

# Etapa 6 — Comparação entre sensores

Como os sensores possuem resoluções e bandas diferentes, é importante analisar suas diferenças.

Comparações:

* resolução espacial
* número de bandas
* variabilidade espectral
* distribuição de valores

Objetivo:

Identificar qual sensor pode fornecer maior poder discriminativo para o problema de detecção de terras raras.

---

# Etapa 7 — Redução de dimensionalidade

Devido ao grande número de features geradas (pixels × bandas), técnicas de redução de dimensionalidade podem ser aplicadas.

Análises possíveis:

* PCA
* t-SNE
* UMAP

Objetivo:

* visualizar clusters
* verificar separação entre classes
* identificar estruturas latentes nos dados

---

# 3. Formulação de Hipóteses

A partir da exploração inicial dos dados, algumas hipóteses podem ser formuladas.

### Hipótese 1

Certas bandas espectrais apresentam maior sensibilidade à presença de minerais de terras raras.

Possível evidência:

* maior variabilidade espectral
* padrões diferenciados nas regiões com ocorrência mineral.

---

### Hipótese 2

Combinações específicas de bandas podem melhorar a separação entre áreas com e sem mineralização.

Isso pode ser investigado através de:

* índices espectrais
* combinações lineares de bandas.

---

### Hipótese 3

Sensores com maior resolução espacial (como Sentinel ou Landsat) podem capturar padrões geológicos mais detalhados do que sensores de resolução mais baixa.

---

### Hipótese 4

Mesmo com grande dimensionalidade, a informação relevante pode estar concentrada em um subconjunto reduzido de pixels ou bandas.

Isso justificaria o uso de:

* seleção de features
* autoencoders
* PCA.

---

# Conclusão da Exploração

A exploração inicial do dataset permitirá compreender as características espectrais e espaciais das imagens utilizadas no projeto SpectraAI. Esses insights serão fundamentais para orientar a escolha de arquiteturas de modelos, estratégias de pré-processamento e técnicas de seleção de features, aumentando a probabilidade de sucesso na identificação automática de áreas com potencial ocorrência de terras raras.


In [6]:
!pip -q install pyarrow fastparquet matplotlib pyarrow scikit-learn

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def csv_to_parquet_stream(csv_path, parquet_path, chunksize=10):
    csv_path = Path(csv_path)
    parquet_path = Path(parquet_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV não encontrado: {csv_path}")


    if parquet_path.exists():
        parquet_path.unlink()
        print(f"Arquivo parquet antigo removido: {parquet_path.name}")

    writer = None
    total_rows = 0

    try:
        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunksize)):
            print(f"Processando chunk {i+1} | shape={chunk.shape}")

            # opcional: reduzir memória
            for col in chunk.columns:
                if col not in ["image_id", "filepath"]:
                    chunk[col] = pd.to_numeric(chunk[col], errors="coerce", downcast="float")

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression="snappy")

            writer.write_table(table)
            total_rows += len(chunk)

            print(f"✅ Total acumulado: {total_rows} linhas")

            del chunk, table
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    print("\n🏁 Conversão finalizada")
    print("Parquet salvo em:", parquet_path)
    print("Tamanho (MB):", parquet_path.stat().st_size / (1024**2))

In [5]:
csv_to_parquet_stream(
    csv_path="/home/cc09-g1/modis",
    parquet_path="/home/cc09-g1/modis.parquet",
    chunksize=5
)

KeyboardInterrupt: 

In [11]:
DATA_DIR = Path("/home/cc09-g1/")

LANDSAT_CSV  = DATA_DIR / "landsat"
MODIS_CSV    = DATA_DIR / "modis"
SENTINEL_CSV = DATA_DIR / "sentinel"

print("LANDSAT:", LANDSAT_CSV.exists(), LANDSAT_CSV)
print("MODIS:", MODIS_CSV.exists(), MODIS_CSV)
print("SENTINEL:", SENTINEL_CSV.exists(), SENTINEL_CSV)

LANDSAT: True /home/cc09-g1/landsat
MODIS: True /home/cc09-g1/modis
SENTINEL: True /home/cc09-g1/sentinel


In [7]:
# Célula 4 — Função para carregar dataset
def load_dataset(csv_path, sample_rows=None):
    """
    sample_rows=None -> carrega tudo
    sample_rows=int  -> carrega apenas uma amostra de linhas
    """
    if sample_rows is None:
        df = pd.read_csv(csv_path)
    else:
        df = pd.read_csv(csv_path, nrows=sample_rows)
    return df

In [8]:
#landsat_df  = load_dataset(LANDSAT_CSV, sample_rows=200)
modis_df    = load_dataset(MODIS_CSV, sample_rows=200)
#sentinel_df = load_dataset(SENTINEL_CSV, sample_rows=200)

#print("LANDSAT:", landsat_df.shape)
print("MODIS:", modis_df.shape)
#print("SENTINEL:", sentinel_df.shape)

MODIS: (200, 458754)


In [ ]:
n_colunas_amostra = 5000

sentinel_df = load_dataset(SENTINEL_CSV, sample_rows=200)

print("SENTINEL:", sentinel_df.shape)

: 

In [9]:
# Primeiras linhas do Dataset

#display(landsat_df.head(2))
display(modis_df.head(2))
#display(sentinel_df.head(2))

,image_id,filepath,b0_p0,b0_p1,b0_p2,b0_p3,b0_p4,b0_p5,b0_p6,b0_p7,...,b6_p65526,b6_p65527,b6_p65528,b6_p65529,b6_p65530,b6_p65531,b6_p65532,b6_p65533,b6_p65534,b6_p65535
0,0,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,1325.0,1096.0,1096.0,1224.0,1171.0,1171.0,1147.0,1147.0,...,1439.0,1430.0,1291.0,1248.0,1248.0,1100.0,1209.0,1209.0,1332.0,1460.0
1,1,/content/drive/MyDrive/SpectraAI/data_raw/MODI...,443.0,460.0,460.0,541.0,541.0,591.0,591.0,593.0,...,539.0,539.0,582.0,619.0,560.0,629.0,805.0,951.0,951.0,1061.0


In [ ]:
# Resumo estatístico do Dataset

def dataset_summary(df, name):
    print("="*100)
    print(f"DATASET: {name}")
    print("="*100)
    print("Shape:", df.shape)
    print("Número de linhas:", df.shape[0])
    print("Número de colunas:", df.shape[1])
    print("Primeiras colunas:", list(df.columns[:10]))
    print("Últimas colunas:", list(df.columns[-10:]))
    print("\nTipos de dados:")
    print(df.dtypes.value_counts())
    print("\nMemória estimada (MB):", df.memory_usage(deep=True).sum() / (1024**2))

In [ ]:
dataset_summary(landsat_df, "LANDSAT")
dataset_summary(modis_df, "MODIS")
dataset_summary(sentinel_df, "SENTINEL")

In [ ]:
def split_metadata_and_features(df):
    metadata_cols = ["image_id", "filepath"]
    feature_cols = [c for c in df.columns if c not in metadata_cols]

    meta = df[metadata_cols].copy()
    X = df[feature_cols].copy()
    return meta, X

In [ ]:
landsat_meta, landsat_X = split_metadata_and_features(landsat_df)
modis_meta, modis_X = split_metadata_and_features(modis_df)
sentinel_meta, sentinel_X = split_metadata_and_features(sentinel_df)

print("LANDSAT X:", landsat_X.shape)
print("MODIS X:", modis_X.shape)
print("SENTINEL X:", sentinel_X.shape)

In [ ]:
def basic_stats(X, name):
    values = X.to_numpy().astype(np.float32)

    print("="*100)
    print(f"ESTATÍSTICAS GERAIS: {name}")
    print("="*100)
    print("Média global:", np.nanmean(values))
    print("Mediana global:", np.nanmedian(values))
    print("Desvio padrão global:", np.nanstd(values))
    print("Mínimo global:", np.nanmin(values))
    print("Máximo global:", np.nanmax(values))
    print("Percentil 1:", np.nanpercentile(values, 1))
    print("Percentil 5:", np.nanpercentile(values, 5))
    print("Percentil 95:", np.nanpercentile(values, 95))
    print("Percentil 99:", np.nanpercentile(values, 99))

In [ ]:
basic_stats(landsat_X, "LANDSAT")
basic_stats(modis_X, "MODIS")
basic_stats(sentinel_X, "SENTINEL")

In [ ]:
def check_missing_and_inf(X, name):
    arr = X.to_numpy()

    print("="*100)
    print(f"VALORES AUSENTES / INVÁLIDOS: {name}")
    print("="*100)
    print("NaN total:", np.isnan(arr).sum())
    print("Inf total:", np.isinf(arr).sum())
    print("Zeros total:", np.sum(arr == 0))
    print("Zeros (%):", 100 * np.mean(arr == 0))

In [ ]:
check_missing_and_inf(landsat_X, "LANDSAT")
check_missing_and_inf(modis_X, "MODIS")
check_missing_and_inf(sentinel_X, "SENTINEL")

In [ ]:
def plot_global_histogram(X, name, bins=100, sample_size=500000):
    arr = X.to_numpy().astype(np.float32).ravel()

    if len(arr) > sample_size:
        idx = np.random.choice(len(arr), size=sample_size, replace=False)
        arr = arr[idx]

    plt.figure(figsize=(10,5))
    plt.hist(arr, bins=bins)
    plt.title(f"Histograma global dos pixels - {name}")
    plt.xlabel("Valor do pixel")
    plt.ylabel("Frequência")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
plot_global_histogram(landsat_X, "LANDSAT")
plot_global_histogram(modis_X, "MODIS")
plot_global_histogram(sentinel_X, "SENTINEL")

In [ ]:
def image_level_stats(X, name):
    arr = X.to_numpy().astype(np.float32)
    row_means = arr.mean(axis=1)
    row_stds = arr.std(axis=1)

    plt.figure(figsize=(10,4))
    plt.plot(row_means, marker='o')
    plt.title(f"Média dos pixels por imagem - {name}")
    plt.xlabel("Índice da imagem")
    plt.ylabel("Média")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(row_stds, marker='o')
    plt.title(f"Desvio padrão dos pixels por imagem - {name}")
    plt.xlabel("Índice da imagem")
    plt.ylabel("Desvio padrão")
    plt.grid(True, alpha=0.3)
    plt.show()

    return row_means, row_stds

In [ ]:
landsat_means, landsat_stds = image_level_stats(landsat_X, "LANDSAT")
modis_means, modis_stds = image_level_stats(modis_X, "MODIS")
sentinel_means, sentinel_stds = image_level_stats(sentinel_X, "SENTINEL")

In [ ]:
comparison_df = pd.DataFrame({
    "LANDSAT_mean": landsat_means,
    "MODIS_mean": modis_means[:len(landsat_means)] if len(modis_means) >= len(landsat_means) else np.pad(modis_means, (0, len(landsat_means)-len(modis_means)), constant_values=np.nan),
    "SENTINEL_mean": sentinel_means[:len(landsat_means)] if len(sentinel_means) >= len(landsat_means) else np.pad(sentinel_means, (0, len(landsat_means)-len(sentinel_means)), constant_values=np.nan),
})

plt.figure(figsize=(8,5))
plt.boxplot([
    landsat_means,
    modis_means,
    sentinel_means
], labels=["LANDSAT", "MODIS", "SENTINEL"])
plt.title("Comparação da média dos pixels por imagem")
plt.ylabel("Média dos pixels")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
PATCH = 256
PIXELS_PER_BAND = PATCH * PATCH

def infer_num_bands(X):
    n_features = X.shape[1]
    num_bands = n_features / PIXELS_PER_BAND
    return n_features, num_bands

for name, X in [("LANDSAT", landsat_X), ("MODIS", modis_X), ("SENTINEL", sentinel_X)]:
    n_features, num_bands = infer_num_bands(X)
    print(f"{name} -> features: {n_features} | bandas inferidas: {num_bands}")

In [ ]:
def reconstruct_image(row_features, patch=256):
    """
    row_features: vetor 1D com todas as features da imagem
    retorno: array (bands, patch, patch)
    """
    row = np.asarray(row_features, dtype=np.float32)
    num_bands = row.shape[0] // (patch * patch)
    img = row.reshape(num_bands, patch, patch)
    return img

In [ ]:
def show_bands(X, index=0, max_bands=6, patch=256, title_prefix="Imagem"):
    img = reconstruct_image(X.iloc[index].values, patch=patch)
    n_bands = img.shape[0]
    n_show = min(n_bands, max_bands)

    plt.figure(figsize=(4*n_show, 4))
    for i in range(n_show):
        plt.subplot(1, n_show, i+1)
        plt.imshow(img[i], cmap="gray")
        plt.title(f"{title_prefix} - Banda {i}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Shape reconstruída:", img.shape)

In [ ]:
show_bands(landsat_X, index=0, max_bands=4, title_prefix="LANDSAT")
show_bands(modis_X, index=0, max_bands=4, title_prefix="MODIS")
show_bands(sentinel_X, index=0, max_bands=4, title_prefix="SENTINEL")

In [ ]:
def show_rgb(X, index=0, rgb_bands=(0,1,2), patch=256, title="RGB"):
    img = reconstruct_image(X.iloc[index].values, patch=patch)

    if max(rgb_bands) >= img.shape[0]:
        print(f"Não foi possível montar RGB. A imagem só possui {img.shape[0]} bandas.")
        return

    rgb = np.stack([img[rgb_bands[0]], img[rgb_bands[1]], img[rgb_bands[2]]], axis=-1)

    # normalização para visualização
    rgb_min = rgb.min()
    rgb_max = rgb.max()
    if rgb_max > rgb_min:
        rgb = (rgb - rgb_min) / (rgb_max - rgb_min)

    plt.figure(figsize=(6,6))
    plt.imshow(rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
show_rgb(landsat_X, index=0, rgb_bands=(0,1,2), title="LANDSAT RGB")
show_rgb(modis_X, index=0, rgb_bands=(0,1,2), title="MODIS RGB")
show_rgb(sentinel_X, index=0, rgb_bands=(0,1,2), title="SENTINEL RGB")

In [ ]:
def mean_per_band(X, patch=256):
    arr = X.to_numpy().astype(np.float32)
    n_samples, n_features = arr.shape
    num_bands = n_features // (patch * patch)

    arr = arr.reshape(n_samples, num_bands, patch, patch)
    band_means = arr.mean(axis=(0,2,3))
    band_stds = arr.std(axis=(0,2,3))
    return band_means, band_stds

In [ ]:
def plot_band_stats(X, name, patch=256):
    band_means, band_stds = mean_per_band(X, patch=patch)
    bands = np.arange(len(band_means))

    plt.figure(figsize=(10,4))
    plt.plot(bands, band_means, marker='o')
    plt.title(f"Média por banda - {name}")
    plt.xlabel("Banda")
    plt.ylabel("Média")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(bands, band_stds, marker='o')
    plt.title(f"Desvio padrão por banda - {name}")
    plt.xlabel("Banda")
    plt.ylabel("Desvio padrão")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
plot_band_stats(landsat_X, "LANDSAT")
plot_band_stats(modis_X, "MODIS")
plot_band_stats(sentinel_X, "SENTINEL")

In [ ]:
def band_correlation_matrix(X, patch=256):
    arr = X.to_numpy().astype(np.float32)
    n_samples, n_features = arr.shape
    num_bands = n_features // (patch * patch)

    arr = arr.reshape(n_samples, num_bands, patch, patch)
    band_vectors = arr.mean(axis=(2,3))  # média espacial de cada banda por imagem
    corr = np.corrcoef(band_vectors, rowvar=False)
    return corr

In [ ]:
def plot_corr_matrix(X, name, patch=256):
    corr = band_correlation_matrix(X, patch=patch)

    plt.figure(figsize=(6,5))
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar()
    plt.title(f"Correlação entre bandas - {name}")
    plt.xlabel("Banda")
    plt.ylabel("Banda")
    plt.show()

In [ ]:
plot_corr_matrix(landsat_X, "LANDSAT")
plot_corr_matrix(modis_X, "MODIS")
plot_corr_matrix(sentinel_X, "SENTINEL")

In [ ]:
def run_pca(X, name, n_components=2, sample_size=100):
    arr = X.to_numpy().astype(np.float32)

    if arr.shape[0] > sample_size:
        idx = np.random.choice(arr.shape[0], size=sample_size, replace=False)
        arr = arr[idx]

    scaler = StandardScaler()
    arr_scaled = scaler.fit_transform(arr)

    pca = PCA(n_components=n_components)
    comps = pca.fit_transform(arr_scaled)

    print(f"{name} - Variância explicada:", pca.explained_variance_ratio_)
    print(f"{name} - Variância acumulada:", pca.explained_variance_ratio_.sum())

    plt.figure(figsize=(7,5))
    plt.scatter(comps[:,0], comps[:,1])
    plt.title(f"PCA - {name}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)
    plt.show()

    return comps, pca

In [ ]:
landsat_pca, landsat_pca_model = run_pca(landsat_X, "LANDSAT", sample_size=min(100, len(landsat_X)))
modis_pca, modis_pca_model = run_pca(modis_X, "MODIS", sample_size=min(100, len(modis_X)))
sentinel_pca, sentinel_pca_model = run_pca(sentinel_X, "SENTINEL", sample_size=min(100, len(sentinel_X)))

In [ ]:
def detect_outliers_by_mean(X, name, z_thresh=3.0):
    arr = X.to_numpy().astype(np.float32)
    row_means = arr.mean(axis=1)

    mu = row_means.mean()
    sigma = row_means.std()

    z = (row_means - mu) / (sigma + 1e-8)
    outliers = np.where(np.abs(z) > z_thresh)[0]

    print("="*100)
    print(f"OUTLIERS POR MÉDIA - {name}")
    print("="*100)
    print("Média das médias:", mu)
    print("Std das médias:", sigma)
    print("Quantidade de outliers:", len(outliers))
    print("Índices:", outliers[:20])

    plt.figure(figsize=(10,4))
    plt.plot(row_means, marker='o')
    plt.axhline(mu + z_thresh*sigma, linestyle='--')
    plt.axhline(mu - z_thresh*sigma, linestyle='--')
    plt.title(f"Outliers por média da imagem - {name}")
    plt.xlabel("Imagem")
    plt.ylabel("Média")
    plt.grid(True, alpha=0.3)
    plt.show()

    return outliers

In [ ]:
landsat_outliers = detect_outliers_by_mean(landsat_X, "LANDSAT")
modis_outliers = detect_outliers_by_mean(modis_X, "MODIS")
sentinel_outliers = detect_outliers_by_mean(sentinel_X, "SENTINEL")

In [ ]:
def summarize_sensor(X, name):
    arr = X.to_numpy().astype(np.float32)
    return {
        "sensor": name,
        "n_images": arr.shape[0],
        "n_features": arr.shape[1],
        "global_mean": float(arr.mean()),
        "global_std": float(arr.std()),
        "global_min": float(arr.min()),
        "global_max": float(arr.max())
    }

summary_table = pd.DataFrame([
    summarize_sensor(landsat_X, "LANDSAT"),
    summarize_sensor(modis_X, "MODIS"),
    summarize_sensor(sentinel_X, "SENTINEL"),
])

display(summary_table)

In [ ]:
def auto_observations(summary_df):
    print("OBSERVAÇÕES INICIAIS")
    print("="*100)

    for _, row in summary_df.iterrows():
        print(f"\nSensor: {row['sensor']}")
        print(f"- Número de imagens analisadas: {row['n_images']}")
        print(f"- Número de features: {row['n_features']}")
        print(f"- Média global dos pixels: {row['global_mean']:.4f}")
        print(f"- Desvio padrão global: {row['global_std']:.4f}")
        print(f"- Faixa de valores: [{row['global_min']:.4f}, {row['global_max']:.4f}]")

auto_observations(summary_table)

In [ ]:
print("""
HIPÓTESES INICIAIS PARA O SPECTRAAI

1. Certos sensores apresentam maior variabilidade espectral do que outros,
   o que pode indicar maior capacidade de discriminar regiões com mineralização.

2. Algumas bandas possuem média e desvio padrão mais informativos,
   sugerindo maior relevância para o treinamento do modelo.

3. A presença de bandas altamente correlacionadas pode indicar redundância,
   abrindo espaço para redução de dimensionalidade ou seleção de features.

4. Imagens com comportamento estatístico muito distinto do restante podem
   representar outliers, erros de aquisição ou regiões geologicamente particulares.

5. Sensores com melhor equilíbrio entre resolução espacial e variabilidade espectral
   podem oferecer melhor desempenho preditivo para detecção de terras raras.

6. Caso surjam agrupamentos no PCA, isso pode indicar que existe estrutura latente
   nos dados que pode ser aproveitada por modelos supervisionados ou não supervisionados.
""")

In [ ]:
eda_summary_path = DATA_DIR / "eda_summary_sensores.csv"
summary_table.to_csv(eda_summary_path, index=False)
print("Resumo salvo em:", eda_summary_path)